# JAX Learning Tutorial: From Basics to Advanced

A comprehensive guide to mastering JAX for scientific computing, sampling algorithms, optimization, and deep learning.

## Topics Covered:
1. Introduction & Installation
2. JAX Basics - Arrays and Operations
3. Functional Transformations (grad, jit, vmap)
4. Linear Algebra Operations
5. Random Number Generation (critical for sampling)
6. Automatic Differentiation (autodiff)
7. Optimization Algorithms
8. Neural Networks with Flax
9. Advanced: Functional Programming Patterns
10. Practical: Building a Simple Sampler

## Section 1: Introduction & Installation

JAX is a Python library for high-performance numerical computing with composable function transformations:
- **grad**: Automatic differentiation
- **jit**: Just-in-time compilation
- **vmap**: Vectorization
- **pmap**: Parallel mapping

Key differences from NumPy:
- JAX arrays are immutable
- JAX operations are functional (no side effects)
- Designed for GPU/TPU acceleration
- JIT compilation for performance

In [ ]:
# Installation (run in terminal if not already installed):
# pip install jax jaxlib flax optax

# Import essential libraries
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, pmap
from jax import random
import numpy as np
import matplotlib.pyplot as plt

print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")
print(f"Default backend: {jax.default_backend()}")

## Section 2: JAX Basics - Arrays and Operations

### 2.1 Creating JAX Arrays

JAX arrays use `DeviceArray` type and support GPU/TPU acceleration. They are immutable.
Key difference: JAX arrays are functional - operations don't modify originals but return new arrays.

In [ ]:
# 2.1 Creating JAX Arrays
x = jnp.array([1.0, 2.0, 3.0, 4.0])
print(f"Array: {x}")
print(f"Type: {type(x)}")
print(f"Shape: {x.shape}")
print(f"Dtype: {x.dtype}")

# Creating arrays with different methods
zeros = jnp.zeros((3, 4))
ones = jnp.ones(5)
arange = jnp.arange(0, 10, 2)
linspace = jnp.linspace(0, 1, 5)
eye = jnp.eye(3)

print(f"\nZeros shape: {zeros.shape}")
print(f"Eye matrix:\n{eye}")

# Most NumPy operations work with jnp
result = jnp.sin(x)
print(f"\nsin(x): {result}")

### 2.2 Immutability and Functional Updates

JAX arrays are IMMUTABLE. You can't modify them in-place.

In [ ]:
# 2.2 Immutability - JAX arrays cannot be modified in-place
x = jnp.array([1.0, 2.0, 3.0])

# DON'T TRY THIS (raises error):
# x[0] = 10.0  # TypeError: 'DeviceArray' object does not support item assignment

# Instead, create a new array using .at[] syntax
x_new = x.at[0].set(10.0)
print(f"Original x: {x}")
print(f"Updated x_new: {x_new}")

# Other useful update operations
x_add = x.at[1].add(5.0)  # Add 5 to index 1
x_mul = x.at[2].multiply(3.0)  # Multiply index 2 by 3
print(f"After add: {x_add}")
print(f"After multiply: {x_mul}")

# Batch updates
indices = jnp.array([0, 2])
values = jnp.array([100.0, 300.0])
x_batch = x.at[indices].set(values)
print(f"Batch update: {x_batch}")

## Section 3: Automatic Differentiation with `grad`

`grad` computes gradients of functions. It's one of JAX's most powerful features.

Key points:
- Derivatives are computed symbolically before execution
- Can differentiate w.r.t. any argument
- Can compose multiple derivatives (2nd order, 3rd order, etc.)
- Perfect for optimization and sampling algorithms

In [ ]:
# 3.1 Basic Gradients

# Simple function
def f(x):
    return x**2 + 2*x + 1

x = 3.0
f_value = f(x)
print(f"f({x}) = {f_value}")

# Compute gradient (derivative)
grad_f = grad(f)  # Returns function that computes gradient
df_dx = grad_f(x)
print(f"df/dx at x={x}: {df_dx}")

# Analytical: d/dx(x^2 + 2x + 1) = 2x + 2 = 2*3 + 2 = 8 ✓

# 3.2 Gradients of multivariate functions
def rosenbrock(x):
    """Rosenbrock function for 2D optimization"""
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

x_vec = jnp.array([0.0, 0.0])
grad_rosenbrock = grad(rosenbrock)
gradient = grad_rosenbrock(x_vec)
print(f"\nGradient of Rosenbrock at {x_vec}: {gradient}")

In [ ]:
# 3.3 Higher-order derivatives (Hessian, 2nd derivatives)
def simple_f(x):
    return jnp.sum(x**3)

x = jnp.array([1.0, 2.0])

# First derivative
grad_f = grad(simple_f)
print(f"First derivative at {x}: {grad_f(x)}")

# Second derivative (Hessian diagonal)
hess_diag = grad(lambda x: jnp.sum(grad_f(x)))
print(f"Hessian diagonal at {x}: {hess_diag(x)}")

# 3.4 Gradients w.r.t. specific arguments
def multi_arg_func(x, y):
    return jnp.sum(x**2 + 2*x*y + y**3)

x = jnp.array([1.0, 2.0])
y = jnp.array([3.0, 4.0])

grad_wrt_x = grad(multi_arg_func, argnums=0)  # gradient w.r.t. x (1st arg)
grad_wrt_y = grad(multi_arg_func, argnums=1)  # gradient w.r.t. y (2nd arg)

print(f"\n∂f/∂x: {grad_wrt_x(x, y)}")
print(f"∂f/∂y: {grad_wrt_y(x, y)}")

# Both simultaneously
grad_both = grad(multi_arg_func, argnums=(0, 1))
print(f"Both gradients: {grad_both(x, y)}")

## Section 4: JIT Compilation with `jit`

`jit` (Just-In-Time) compilation traces your function and compiles it to run efficiently on GPU/TPU.

Performance impact:
- First call: slower (compilation overhead)
- Subsequent calls: much faster (10-100x speedup)
- Essential for tight loops in optimization and sampling

In [ ]:
# 4.1 Basic JIT Example
import time

def slow_func(x):
    """Function without JIT"""
    total = 0.0
    for i in range(1000):
        total = total + jnp.sin(x[i % len(x)])
    return total

def fast_func(x):
    """Function with JIT"""
    return jnp.sum(jnp.sin(x))

# Compile JIT version
fast_func_jit = jit(fast_func)

x = jnp.ones(100)

# Warm up (compilation happens here)
_ = fast_func_jit(x)

# Time comparison
n_iter = 1000
start = time.time()
for _ in range(n_iter):
    _ = fast_func(x)
time_normal = time.time() - start

start = time.time()
for _ in range(n_iter):
    _ = fast_func_jit(x)
time_jit = time.time() - start

print(f"Time without JIT: {time_normal:.4f}s")
print(f"Time with JIT: {time_jit:.4f}s")
print(f"Speedup: {time_normal / time_jit:.2f}x")

# 4.2 JIT with Grad
@jit
def loss_func(x):
    return jnp.sum((x - 3.0)**2)

grad_jit = jit(grad(loss_func))
x = jnp.array([1.0, 2.0, 3.0])
gradient = grad_jit(x)
print(f"\nJIT compiled gradient: {gradient}")

## Section 5: Vectorization with `vmap`

`vmap` (vectorized map) automatically vectorizes functions over a batch dimension.

Advantages:
- Write code for a single example
- Get batch processing automatically
- No manual loops
- Works with GPU/TPU efficiently
- Critical for MCMC sampling (batch chains)

In [ ]:
# 5.1 Basic vmap Example
def single_sample_log_prob(x, mean):
    """Compute log probability for a single sample"""
    return -0.5 * jnp.sum((x - mean)**2)

# Vectorize over batch of samples (first axis)
batch_log_prob = vmap(single_sample_log_prob, in_axes=(0, None))

# Create batch of samples
batch_x = jnp.array([
    [1.0, 2.0, 3.0],
    [1.5, 2.5, 3.5],
    [2.0, 3.0, 4.0]
])
mean = jnp.array([2.0, 3.0, 4.0])

# Run vmap
log_probs = batch_log_prob(batch_x, mean)
print(f"Batch log probs shape: {log_probs.shape}")
print(f"Batch log probs: {log_probs}")

# 5.2 Multiple batch dimensions
def loss_single(x, y):
    return jnp.sum((x - y)**2)

# Vectorize over 2D array of x values and 2D array of y values
batch_x_2d = jnp.ones((5, 3, 2))  # 5 batches, 3 samples, 2 features
batch_y_2d = jnp.ones((5, 3, 2)) * 2.0

batch_loss = vmap(vmap(loss_single))  # nested vmap
losses = batch_loss(batch_x_2d, batch_y_2d)
print(f"\nBatch loss shape: {losses.shape}")
print(f"Batch loss:\n{losses}")

## Section 6: Random Number Generation (Critical for Sampling!)

JAX's random number generation uses functional/pure approach - no global state!

Key differences from NumPy:
- Need explicit PRNGKey
- Split keys for independent streams
- Fully deterministic and reproducible
- Essential for sampling algorithms

In [ ]:
# 6.1 Creating and splitting keys
key = random.PRNGKey(0)
print(f"PRNG Key: {key}")
print(f"Key shape: {key.shape}")

# Generate random numbers
subkey, key = random.split(key)  # Split for independent random stream
x = random.normal(subkey, shape=(3, 3))
print(f"\nNormal random samples:\n{x}")

# Multiple splits for multiple operations
key = random.PRNGKey(42)
subkey1, subkey2, subkey3, key = random.split(key, 4)

print(f"\nGaussian: {random.normal(subkey1, shape=(2,))}")
print(f"Uniform: {random.uniform(subkey2, shape=(2,))}")
print(f"Exponential: {random.exponential(subkey3, shape=(2,))}")

# 6.2 Reproducibility - same seed gives same results
key1 = random.PRNGKey(0)
sample1 = random.normal(key1, shape=(3,))

key2 = random.PRNGKey(0)
sample2 = random.normal(key2, shape=(3,))

print(f"\nSample 1: {sample1}")
print(f"Sample 2: {sample2}")
print(f"Are they equal? {jnp.allclose(sample1, sample2)}")

# 6.3 Sampling from distributions (for MCMC proposals)
key = random.PRNGKey(0)
subkey, key = random.split(key)

# Multivariate normal
mean = jnp.array([0.0, 0.0])
cov = jnp.eye(2)
sample = random.multivariate_normal(subkey, mean, cov)
print(f"\nMultivariate normal sample: {sample}")

## Section 7: Linear Algebra Operations

JAX provides efficient linear algebra via `jax.numpy.linalg`

Essential for:
- Matrix operations in sampling
- Covariance matrix computations
- Solving linear systems
- Matrix decompositions

In [ ]:
# 7.1 Basic matrix operations
from jax import linalg

A = jnp.array([[1.0, 2.0], [3.0, 4.0]])
B = jnp.array([[2.0, 0.0], [1.0, 2.0]])

# Matrix multiplication
C = jnp.dot(A, B)  # or A @ B
print(f"A @ B =\n{C}")

# Determinant
det_A = linalg.det(A)
print(f"\ndet(A) = {det_A}")

# Matrix inverse
A_inv = linalg.inv(A)
print(f"A^-1 =\n{A_inv}")

# Verify A @ A^-1 = I
print(f"A @ A^-1 =\n{A @ A_inv}")

# 7.2 Matrix decompositions (critical for sampling!)
A = jnp.array([[3.0, 1.0], [1.0, 2.0]])  # Symmetric positive definite

# Cholesky decomposition (for sampling from Gaussians)
L = linalg.cholesky(A)
print(f"\nCholesky L =\n{L}")
print(f"L @ L^T =\n{L @ L.T}")  # Should equal A

# Eigendecomposition
eigenvalues, eigenvectors = linalg.eigh(A)
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors:\n{eigenvectors}")

# 7.3 Solving linear systems
A = jnp.array([[3.0, 1.0], [1.0, 2.0]])
b = jnp.array([5.0, 3.0])

# Solve Ax = b
x = linalg.solve(A, b)
print(f"\nSolution to Ax = b: {x}")
print(f"Verify A @ x = b: {A @ x}")

## Section 8: Optimization with Optax

Optax is a gradient-based optimization library that works seamlessly with JAX.

Key optimizers:
- SGD, Adam, RMSprop, etc.
- Learning rate schedules
- Gradient clipping, weight decay
- Perfect for training neural networks and fitting models

In [ ]:
# 8.1 Basic optimization loop with optax
try:
    import optax
except:
    print("Installing optax...")
    import subprocess
    subprocess.check_call(["pip", "install", "optax"])
    import optax

# Define loss function
def loss_fn(params):
    x, y = params
    return (x - 3.0)**2 + (y + 2.0)**2  # Minimum at (3, -2)

# Initialize parameters and optimizer
params = jnp.array([0.0, 0.0])
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(params)

# Training loop
print("Optimization iterations:")
for step in range(10):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    print(f"Step {step}: loss={loss:.4f}, params={params}")

print(f"\nFinal optimum: {params} (target: [3, -2])")

# 8.2 Training with gradient descent  
@jit
def update_step(params, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Reset
params = jnp.array([0.0, 0.0])
opt_state = optimizer.init(params)
optimizer = optax.sgd(learning_rate=0.1)
opt_state = optimizer.init(params)

print("\nJIT-compiled optimization:")
for step in range(5):
    params, opt_state, loss = update_step(params, opt_state)
    print(f"Step {step}: loss={loss:.4f}")

## Section 9: Neural Networks with Flax

Flax is a neural network library built on JAX.

Key concepts:
- Modules define network architecture
- `apply` method runs the network
- Parameters are immutable (functional style)
- Works perfectly with JAX transformations

In [ ]:
# 9.1 Simple neural network with Flax
try:
    from flax import linen as nn
except:
    print("Installing flax...")
    import subprocess
    subprocess.check_call(["pip", "install", "flax"])
    from flax import linen as nn

# Define a simple MLP
class MLP(nn.Module):
    features: list  # List of layer sizes
    
    def setup(self):
        self.layers = [nn.Dense(feat) for feat in self.features]
    
    def __call__(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = nn.relu(x)
        x = self.layers[-1](x)
        return x

# Create model and initialize parameters
model = MLP(features=[64, 32, 10])
key = random.PRNGKey(0)
x_dummy = jnp.ones((1, 20))  # Batch of 1, input dimension 20

# Initialize parameters
params = model.init(key, x_dummy)
print(f"Model parameters initialized")
print(f"Parameter keys: {params.keys()}")

# Forward pass
output = model.apply(params, x_dummy)
print(f"\nOutput shape: {output.shape}")
print(f"Output sample:\n{output}")

# 9.2 Training a simple network
def loss_function(params, x, y):
    predictions = model.apply(params, x)
    return jnp.mean((predictions - y)**2)

# Create dummy training data
key = random.PRNGKey(1)
subkey, key = random.split(key)
x_train = random.normal(subkey, shape=(32, 20))
y_train = random.normal(key, shape=(32, 10))

# Optimizer
optimizer = optax.adam(learning_rate=0.001)
opt_state = optimizer.init(params)

# Training step
@jit
def train_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(loss_function)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Train for a few steps
print("\nTraining:")
for step in range(5):
    params, opt_state, loss = train_step(params, opt_state, x_train, y_train)
    print(f"Step {step}: loss={loss:.4f}")

## Section 10: Advanced - Composing Transformations

One of JAX's superpowers: composing `jit`, `grad`, `vmap` together!

This is essential for efficient sampling algorithms:

In [ ]:
# 10.1 Composing jit + grad
def loss_fn(params, x, y):
    predictions = jnp.dot(params, x)
    return jnp.mean((predictions - y)**2)

# Compose JIT with grad
grad_loss_jit = jit(grad(loss_fn, argnums=0))

params = jnp.array([1.0, 2.0, 3.0])
x = jnp.array([0.5, 0.4, 0.3])
y = jnp.array([1.5])

# This is compiled!
gradient = grad_loss_jit(params, x, jnp.array([2.0]))
print(f"JIT-compiled gradient: {gradient}")

# 10.2 vmap + grad for batch gradients
def single_loss(params, x, y):
    pred = jnp.dot(params, x)
    return (pred - y)**2

# Vectorize gradient computation over batch
batch_grad = vmap(grad(single_loss), in_axes=(None, 0, 0))

params = jnp.array([1.0, 2.0])
batch_x = jnp.array([
    [0.1, 0.2],
    [0.3, 0.4],
    [0.5, 0.6]
])
batch_y = jnp.array([1.0, 2.0, 3.0])

gradients = batch_grad(params, batch_x, batch_y)
print(f"\nBatch gradients shape: {gradients.shape}")
print(f"Batch gradients:\n{gradients}")

# 10.3 Practical: Metropolis-Hastings step with all transformations
def log_prob(x):
    """Log probability function (unnormalized)"""
    return -0.5 * jnp.sum(x**2)

def metropolis_hastings_step(key, current_x, log_prob_fn, proposal_std=0.1):
    """Single MCMC step with jit-compiled acceptance"""
    # Propose new state
    subkey, key = random.split(key)
    proposed_x = current_x + proposal_std * random.normal(subkey, shape=current_x.shape)
    
    # Compute log acceptance ratio
    log_alpha = log_prob_fn(proposed_x) - log_prob_fn(current_x)
    
    # Accept or reject
    subkey, key = random.split(key)
    accept = jnp.log(random.uniform(subkey)) < log_alpha
    
    # Update state
    new_x = jnp.where(accept, proposed_x, current_x)
    return new_x, key, accept

# Compile for speed
mh_step_jit = jit(metropolis_hastings_step)

# Run a short chain
key = random.PRNGKey(0)
x = jnp.array([0.0, 0.0])

print("\nMCMC chain (first 10 steps):")
for i in range(10):
    x, key, accept = mh_step_jit(key, x, log_prob)
    print(f"Step {i}: x={x}, accept={accept}")

# Vectorize over multiple chains
batch_mh_step = vmap(
    lambda key, x: metropolis_hastings_step(key, x, log_prob),
    in_axes=(0, 0)
)

# Multiple chains in parallel
key = random.PRNGKey(0)
keys = random.split(key, 4)
batch_x = jnp.zeros((4, 2))  # 4 chains, 2D state

batch_x_new, keys_new, accepts = batch_mh_step(keys, batch_x)
print(f"\nParallel chains shape: {batch_x_new.shape}")

## Section 11: Best Practices for Sampling Algorithms

### Key Takeaways for Building Samplers with JAX:

1. **Pure Functional Style**
   - Don't mutate state, return new values
   - Use `.at[]` for array updates
   - Pass PRNGKey explicitly

2. **Use JIT Compilation**
   - Wrap tight loops with `@jit`
   - Dramatic speedups (10-100x)
   - Compile once, run many times

3. **Vectorization for Multiple Chains**
   - Use `vmap` for parallel chains
   - Efficient GPU/TPU utilization
   - Natural batching for ensemble methods

4. **Gradients for Efficient Sampling**
   - Use `grad` for Hamiltonian methods
   - Langevin dynamics
   - Variational inference

5. **Random Number Generation**
   - Split keys for independent streams
   - Deterministic and reproducible
   - Critical for MCMC debugging

6. **Memory Efficiency**
   - Stack computations when possible
   - Use in-place updates with `.at[]`
   - Be careful with large batch dimensions

## Section 12: Practical Example - Bayesian Linear Regression with HMC

Combines everything: gradients, linear algebra, random numbers, and optimization.

In [ ]:
# 12.1 Generate synthetic data
key = random.PRNGKey(42)
n_samples = 50
n_features = 3

# True parameters
true_w = jnp.array([1.5, -2.0, 0.5])
true_sigma = 0.1

# Generate data
subkey, key = random.split(key)
X = random.normal(subkey, shape=(n_samples, n_features))

subkey, key = random.split(key)
y = X @ true_w + true_sigma * random.normal(subkey, shape=(n_samples,))

print(f"Data shape: X={X.shape}, y={y.shape}")

# 12.2 Bayesian linear regression - posterior over weights
def log_posterior(w, X, y, prior_std=1.0):
    """Log posterior: log p(w|X,y) = log p(y|X,w) + log p(w)"""
    # Likelihood
    predictions = X @ w
    sigma = 0.1
    log_likelihood = -0.5 * jnp.sum(((y - predictions) / sigma)**2)
    
    # Prior
    log_prior = -0.5 * jnp.sum((w / prior_std)**2)
    
    return log_likelihood + log_prior

# 12.3 Gradient-based sampling (simplified Langevin dynamics)
def langevin_step(key, w, X, y, step_size=0.01):
    """One step of Langevin dynamics: w_new = w + 0.5*step_size*grad_log_p + sqrt(step_size)*z"""
    subkey, key = random.split(key)
    
    # Compute gradient of log posterior
    grad_log_p = grad(log_posterior)(w, X, y)
    
    # Langevin update
    z = random.normal(subkey, shape=w.shape)
    w_new = w + 0.5 * step_size * grad_log_p + jnp.sqrt(step_size) * z
    
    return w_new, key

# Compile for speed
langevin_step_jit = jit(langevin_step)

# Run sampling
w_samples = []
w = jnp.zeros(n_features)
key = random.PRNGKey(0)

print("\nLangevin sampling (first 5 iterations):")
for i in range(100):
    w, key = langevin_step_jit(key, w, X, y)
    if i % 20 == 0:
        print(f"Step {i}: w={w}, log_post={log_posterior(w, X, y):.2f}")
    if i >= 50:  # Collect samples after burn-in
        w_samples.append(w)

w_samples = jnp.array(w_samples)

print(f"\nSampled weights mean: {jnp.mean(w_samples, axis=0)}")
print(f"True weights: {true_w}")
print(f"Sampled weights std: {jnp.std(w_samples, axis=0)}")

# 12.4 Visualization of posterior
print("\nPosterior summary:")
for i in range(n_features):
    mean = jnp.mean(w_samples[:, i])
    std = jnp.std(w_samples[:, i])
    print(f"w[{i}]: {mean:.3f} ± {std:.3f}")

## Section 13: Summary & Next Steps

### What We Covered:
✓ JAX basics and array operations  
✓ Automatic differentiation (`grad`)  
✓ JIT compilation (`jit`)  
✓ Vectorization (`vmap`)  
✓ Random number generation (Pure functional RNG)  
✓ Linear algebra operations  
✓ Optimization with Optax  
✓ Neural networks with Flax  
✓ Composing transformations  
✓ Practical sampling example  

### Key JAX Concepts to Remember:
1. **Functional Programming**: Arrays are immutable, functions are pure
2. **Transformations**: Functions that transform functions (`grad`, `jit`, `vmap`)
3. **Composability**: Chain transformations for powerful effects
4. **GPU/TPU Ready**: Automatic device placement, SPMD-friendly design
5. **Deterministic**: Explicit random number handling for reproducibility

### Recommended Next Steps:

**For Sampling Algorithms:**
- Study Hamiltonian Monte Carlo (HMC)
- Learn about No-U-Turn Sampler (NUTS)
- Explore variational inference

**For Deep Learning:**
- Build more complex networks with Flax
- Learn about batch normalization, dropout
- Study attention mechanisms

**For Linear Algebra:**
- Study matrix decompositions in depth
- Learn SVD, QR, Cholesky for different applications
- Understand spectral methods

**For Optimization:**
- Explore learning rate schedules
- Learn about adaptive methods (Adam, RMSprop)
- Study gradient clipping and normalization

### Useful Resources:
- JAX Documentation: https://jax.readthedocs.io/
- JAX GitHub: https://github.com/google/jax
- Flax Documentation: https://flax.readthedocs.io/
- Optax Documentation: https://optax.readthedocs.io/
- Tutorial example: jax.readthedocs.io/en/latest/notebooks/thinking_in_jax.html

## Appendix: Practice Exercises

### Easy:
1. **Matrix Trace**: Write a function to compute the trace of a matrix, then use `grad` to get its gradient w.r.t. input
2. **Batch Normalization**: Compute mean and std for a batch, apply normalization with `vmap`

### Medium:
3. **Custom Optimizer**: Implement momentum SGD using JAX arrays and immutable updates
4. **Sampling from Gaussian Mixture**: Use `vmap` and `random` module to sample from multiple Gaussians in parallel

### Advanced:
5. **Implement Hamiltonian Monte Carlo**: Full HMC with leapfrog integrator and acceptance check
6. **Neural ODE**: Build a simple continuous ODE solver integrated with a neural network
7. **Variational Autoencoder**: Implement VAE with Flax and optimize with Optax